# TF-IDF

Based on Term Frequency/Inverted Index last time

The text files are placed in hdfs folder '/input', i.e. we have several files in the folder.

The wholeTextFiles will read the files, and output a tuple (filename, fulltext)

data (i.e., wholeTextFiles) -> (filename, fulltext)

Here we calculate the tf "Term Frequency":
**tf = 1+log10(tf)**

In [1]:
import re
def removeSomeTexts(text):
    return re.sub(r"[^a-z-'\s]",'', text)

In [2]:
from pyspark import SparkConf, SparkContext
import os, re
from pyspark.sql.session import SparkSession
import math

spark = SparkSession.builder.master('local').appName('MyTf-Idf').getOrCreate()
sc = spark.sparkContext
#sc = SparkContext.getOrCreate(SparkConf())

sc._conf.set('spark.executor.memory', '8g')
sc._conf.set('spark.driver.memory', '8g')
sc._conf.set('spark.driver.maxResultsSize', '0')

# you need to start hadoop first!!!
# start-dfs.sh
# start-yarn.sh
# make sure you have put data files from your local to /hdfs/input
# hdfs dfs -mkdir /hdfs
# hdfs dfs -mkdir /input
# hdfs dfs -put  yourlocalpath/*.txt  /hdfs/input
# double check the file are uploaded or not
# hdfs dfs -ls /hdfs/input
#data (i.e., wholeTextFiles) -> (filename, fulltext)
#data = sc.wholeTextFiles('hdfs://localhost:9000/hdfs/input')  # small 13.7MB, it is ok
#data = sc.wholeTextFiles('hdfs://localhost:9000/hdfs/book1g') # large around 1G, failed
#data = sc.wholeTextFiles('hdfs://localhost:9000/hdfs/book500m') # medium around 605M, failed
data = sc.wholeTextFiles('hdfs://localhost:9000/hdfs/book100m') # medium around 108M, ok

numFiles = data.count() 
print(numFiles)

175


In [3]:
wordcount = data.flatMap(lambda x: [((os.path.basename(x[0]), removeSomeTexts(i.lower())), 1) for i in re.split('\W+', x[1])]) \
        .reduceByKey(lambda a, b: a + b)\
        .map(lambda x: (x[0][1],(x[0][0],x[1])))\
        .filter(lambda x: x[0] != '')
wordcount.take(10)   

[('tells', ('1.txt', 2)),
 ('means', ('1.txt', 8)),
 ('explanatory', ('1.txt', 2)),
 ('was', ('1.txt', 25)),
 ('decent', ('1.txt', 2)),
 ('their', ('1.txt', 81)),
 ('liberty', ('1.txt', 12)),
 ('fatiguing', ('1.txt', 2)),
 ('cutting', ('1.txt', 2)),
 ('introducing', ('1.txt', 2))]

In [4]:
# tf = 1+log10(tf)
tf = wordcount.map(lambda x: (x[0],(x[1][0], 1+math.log10((x[1][1]) ))))
tf.take(10)

[('tells', ('1.txt', 1.3010299956639813)),
 ('means', ('1.txt', 1.9030899869919435)),
 ('explanatory', ('1.txt', 1.3010299956639813)),
 ('was', ('1.txt', 2.3979400086720375)),
 ('decent', ('1.txt', 1.3010299956639813)),
 ('their', ('1.txt', 2.9084850188786495)),
 ('liberty', ('1.txt', 2.079181246047625)),
 ('fatiguing', ('1.txt', 1.3010299956639813)),
 ('cutting', ('1.txt', 1.3010299956639813)),
 ('introducing', ('1.txt', 1.3010299956639813))]

## Computing Inverse Document Frequency (IDF)

The IDF score tells us how important is a particular word in the whole corpus. For instance, if a word is appearing in each document then the IDF score will be zero for that.

We know that the total number of documents for our case is 3. So, our next step will be to compute how many documents contain a particular token w. Further, we can use this information to calculate the IDF scores for all the tokens in the corpus.

We will be chaining the previous map/reduce computation to calculate IDF in this section. After the reduced step in the previous section, we obtained key/value pairs for respective tokens along with its term frequency.

In the first step, we will map the previous key/value pair to a new key/value pair. Here, the key will the token and its value will be the document id TF for that token along with a counter of 1. This one here indicates the presence of a word in document id associated with that.

 ((token, (filename, TF)) -> (token, 1)

Then, we will reduce by key to obtain the count of documents containing a particular token w (wordcount).

(token, 1) -> (token, 1+1+1+...)

In the last step, since we have the number of documents containing each token w, we will just map this final output with the logarithmic transformation to compute the IDF score.

(token, doc-freq-containing-token) -> (token, IDF)

In [5]:
idf = wordcount.map(lambda x: (x[0], 1))\
        .reduceByKey(lambda x, y: x + y)\
        .map(lambda x: (x[0], math.log10(numFiles / (x[1]))))
idf.take(10)

[('tells', 0.18234020833268277),
 ('means', 0.0),
 ('explanatory', 0.7659167939666319),
 ('was', 0.0024888004036947415),
 ('decent', 0.3917796999672191),
 ('their', 0.02032157753871113),
 ('liberty', 0.1748521869401328),
 ('fatiguing', 0.9877655435829883),
 ('cutting', 0.23443787692437684),
 ('introducing', 0.599585372200107)]

## Computing TF-IDF

Now we have two RDDs one having the document id along with its document term frequency for each token. Another RDD having IDF scores for each token.

First, we will perform an inner join to assign each token with a document id, TF, and IDF score.

Second, we will map this function to multiply TF and IDF values of each token associated with respective document id.

Third, we sort the key

New Key/Value pair -> (filename, (token, TF-IDF))

In [6]:
# failed for the size is 600MB
# put it together
tfidf = tf.join(idf).map(lambda x: (x[1][0][0], (x[0], x[1][0][1] * x[1][1]))).sortByKey()\
        .map(lambda x:(x[1][0], (x[0], x[1][1]))).filter(lambda x: x[0]!='')\
        .sortByKey()
tfidf.take(50)

[('a', ('1.txt', 0.0)),
 ('a', ('10.txt', 0.0)),
 ('a', ('100.txt', 0.0)),
 ('a', ('101.txt', 0.0)),
 ('a', ('102.txt', 0.0)),
 ('a', ('103.txt', 0.0)),
 ('a', ('104.txt', 0.0)),
 ('a', ('105.txt', 0.0)),
 ('a', ('106.txt', 0.0)),
 ('a', ('107.txt', 0.0)),
 ('a', ('108.txt', 0.0)),
 ('a', ('109.txt', 0.0)),
 ('a', ('11.txt', 0.0)),
 ('a', ('110.txt', 0.0)),
 ('a', ('111.txt', 0.0)),
 ('a', ('112.txt', 0.0)),
 ('a', ('113.txt', 0.0)),
 ('a', ('115.txt', 0.0)),
 ('a', ('116.txt', 0.0)),
 ('a', ('117.txt', 0.0)),
 ('a', ('118.txt', 0.0)),
 ('a', ('119.txt', 0.0)),
 ('a', ('12.txt', 0.0)),
 ('a', ('120.txt', 0.0)),
 ('a', ('121.txt', 0.0)),
 ('a', ('122.txt', 0.0)),
 ('a', ('123.txt', 0.0)),
 ('a', ('124.txt', 0.0)),
 ('a', ('125.txt', 0.0)),
 ('a', ('126.txt', 0.0)),
 ('a', ('128.txt', 0.0)),
 ('a', ('13.txt', 0.0)),
 ('a', ('130.txt', 0.0)),
 ('a', ('131.txt', 0.0)),
 ('a', ('132.txt', 0.0)),
 ('a', ('133.txt', 0.0)),
 ('a', ('134.txt', 0.0)),
 ('a', ('135.txt', 0.0)),
 ('a', ('136.txt',

In [7]:
#save tfidf index as parquet format 
lines = tfidf.map(lambda x: (x[0], x[1][0], x[1][1])).toDF()

# save it in hdfs file system
# If you have already this directory, delete it first
# hdfs dfs -rm -r /hdfs/output/tfidf_index
#lines.write.save('hdfs://localhost:9000/hdfs/output/tfidf_index')

# you can also save it in your local file system
# If you already have this directory, delete it first
lines.write.save("./data/tfidf_100m")